In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os

from statsmodels.tsa.stattools import adfuller
from hurst import compute_Hc


In [2]:
import sys
sys.path.append(os.path.abspath('/home/juan/haku/tools'))

from basic_functions import get_crypto_data, get_stats, get_stocks_data

In [3]:
data = get_crypto_data('BTCUSDT', '5m')

In [4]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [5]:
data

,Open,High,Low,Close,Volume
Date,,,,,
2017-08-17 04:05:00,4261.480,4261.480,4261.480,4261.480,0.000
2017-08-17 04:10:00,4261.480,4261.480,4261.480,4261.480,0.000
2017-08-17 04:15:00,4261.480,4264.880,4261.480,4261.480,0.485
2017-08-17 04:20:00,4264.880,4266.290,4264.880,4266.290,2.329
2017-08-17 04:25:00,4266.290,4270.410,4261.320,4261.450,6.307
...,...,...,...,...,...
2024-07-31 23:35:00,64730.760,64734.950,64682.020,64688.000,40.579
2024-07-31 23:40:00,64688.000,64764.970,64666.000,64756.580,48.155
2024-07-31 23:45:00,64756.580,64765.780,64650.010,64676.000,144.383


In [6]:
from statsmodels.tsa.stattools import adfuller

# Apply the Augmented Dickey-Fuller Test
resultado_adf = adfuller(data['Close'], maxlag=0)

# Print the results
estadistico_adf = resultado_adf[0]
valor_p = resultado_adf[1]
valores_criticos = resultado_adf[4]

# Print ADF results
print('ADF Statistic:', estadistico_adf)
print('p-value:', valor_p)
print('Critical values:')
for key, value in valores_criticos.items():
    print(f'   {key}: {value}')

# Evaluate stationarity
if valor_p < 0.05:
    print("\nThe series is stationary since the p-value is less than 0.05.")
else:
    print("\nThe series is not stationary since the p-value is greater than 0.05.")

# Analyze if mean reversion is likely
if estadistico_adf < valores_criticos['5%']:
    print("\nThe series seems to show mean reversion tendencies, as the ADF statistic is less than the critical value at the 5% level.")
else:
    print("\nThe series shows no evidence of mean reversion, as the ADF statistic is greater than the critical value at the 5% level.")


ADF Statistic: -0.9400399212695849
p-value: 0.7745009841873411
Critical values:
   1%: -3.430358958320313
   5%: -2.8615439594748846
   10%: -2.5667721074833665

The series is not stationary since the p-value is greater than 0.05.

The series shows no evidence of mean reversion, as the ADF statistic is greater than the critical value at the 5% level.


### Explanation:

- **Apply the Augmented Dickey-Fuller Test**: The ADF test checks if a time series is stationary.
- **ADF Statistic**: The test statistic. A more negative value suggests a stronger rejection of the null hypothesis (non-stationarity).
- **p-value**: If less than 0.05, it indicates the series is stationary.
- **Critical values**: Used to compare the test statistic at different significance levels (1%, 5%, 10%).
- **Stationarity Evaluation**: The series is considered stationary if the p-value is less than 0.05.
- **Mean Reversion Analysis**: If the ADF statistic is lower than the 5% critical value, the series may exhibit mean reversion behavior.


In [7]:
# Calculate the Hurst exponent (h), the constant (c), and d
h, c, d = compute_Hc(data['Close'], kind='price', simplified=False)

# Print the results
print(f'Hurst Exponent (h): {h}')
print(f'Constant (c): {c}')

# Evaluate the behavior of the time series based on the Hurst exponent (h)
if h < 0.5:
    print("\nThe value of the Hurst exponent is less than 0.5, which indicates that the series exhibits mean reversion behavior.")
elif h == 0.5:
    print("\nThe value of the Hurst exponent is equal to 0.5, which indicates random walk behavior.")
else:
    print("\nThe value of the Hurst exponent is greater than 0.5, which indicates persistent trend behavior.")

# Evaluation of the constant c
if c < 0:
    print("\nThe constant c is negative, which could suggest a tendency toward mean reversion.")
else:
    print("\nThe constant c is positive, which could indicate trend-following behavior.")


Hurst Exponent (h): 0.530179783066175
Constant (c): 0.900582353952895

The value of the Hurst exponent is greater than 0.5, which indicates persistent trend behavior.

The constant c is positive, which could indicate trend-following behavior.


### Explanation:

- **Hurst Exponent (h)**:
  - If **h < 0.5**, the series exhibits mean reversion, meaning the values tend to revert to a central average over time.
  - If **h = 0.5**, it indicates a random walk, meaning the series follows a non-predictable path without a clear trend.
  - If **h > 0.5**, the series has a persistent trend behavior, meaning the values tend to follow a directional trend over time.

- **Constant (c)**:
  - If **c < 0**, it may suggest that the series is more likely to exhibit mean reversion behavior.
  - If **c > 0**, it could indicate that the series shows a trend-following behavior.


In [8]:
# Calculate the mean reversion rate
mean_reversion_rate = -np.log(2) / resultado_adf[0]

# Calculate the half-life
half_life = np.log(2) / mean_reversion_rate

# Print the half-life value
print("Half-life:", half_life)

Half-life: 0.9400399212695849


### Explanation:

- **Mean Reversion Rate**:
  - The mean reversion rate is calculated using the formula:

  $$
  \text{Mean Reversion Rate} = \frac{-\ln(2)}{\text{ADF Statistic}}
  $$

  where `resultado_adf[0]` is the ADF statistic from the Augmented Dickey-Fuller test. This rate tells you how quickly the series tends to revert to the mean.

- **Half-life (Vida Media)**:
  - The half-life refers to the time it takes for the series to revert to half of its mean reversion value. It is calculated using the formula:

  $$
  \text{Half-life} = \frac{\ln(2)}{\text{Mean Reversion Rate}}
  $$

  where `np.log(2)` is the natural logarithm of 2. The half-life gives you a sense of how fast or slow the reversion to the mean will occur.

- **Output**:
  - The program prints the half-life, which tells you how long it would take for the series to revert by half towards its mean reversion level.
